## How to Combine GPT-oss with RAG 

The tutorial is developed based on openai-cookbook [example 1](https://github.com/openai/openai-cookbook/blob/main/examples/fine-tuned_qa/ft_retrieval_augmented_generation_qdrant.ipynb) and [example 2](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_combine_GPT4o_with_RAG_Outfit_Assistant.ipynb).

The aim of this notebook is to walk through a example of how to do Retrieval Augmented Generation (RAG).

We've selected a subset of the [SQuAD](https://rajpurkar.github.io/SQuAD-explorer/) dataset, which is a collection of questions and answers about Wikipedia articles. 

We'll cover the following steps:

1. Data Preparation: SQuADv2 Dataset
2. RAG Prompt
   
With GPT-oss + RAG, the goal is to leverage the strengths of both generative and retrieval-based AI techniques.

### Data Preparation: SQuADv2 Data Subsets

For the purpose of demonstration, we'll make small slices from the train and validation splits of the [SQuADv2](https://rajpurkar.github.io/SQuAD-explorer/) dataset. This dataset has questions and contexts where the answer is not present in the context, to help us evaluate how LLM handles this case.

We'll read the data from the JSON files and create a dataframe with the following columns: `question`, `context`, `answer`, `is_impossible`.

#### Download the Data

In [1]:
# !mkdir -p $SCRATCH/data/training
# !wget https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json -O $SCRATCH/data/training/train.json
# !wget https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json -O $SCRATCH/data/training/dev.json

#### Environment Setup

In [1]:
import json
import os
import time

import pandas as pd
from collections import defaultdict
import numpy as np
from tqdm import tqdm

import warnings

tqdm.pandas()
%matplotlib inline
%env HF_HOME={os.environ['SCRATCH']}/HFHOME

env: HF_HOME=/scratch/07980/sli4/HFHOME


#### Read JSON to DataFrame

In [3]:
def json_to_dataframe_with_titles(json_data):
    qas = []
    context = []
    is_impossible = []
    answers = []
    titles = []

    for article in json_data['data']:
        title = article['title']
        for paragraph in article['paragraphs']:
            for qa in paragraph['qas']:
                qas.append(qa['question'].strip())
                context.append(paragraph['context'])
                is_impossible.append(qa['is_impossible'])
                
                ans_list = []
                for ans in qa['answers']:
                    ans_list.append(ans['text'])
                answers.append(ans_list)
                titles.append(title)

    df = pd.DataFrame({'title': titles, 'question': qas, 'context': context, 'is_impossible': is_impossible, 'answers': answers})
    return df

def get_diverse_sample(df, sample_size=100, random_state=42):
    """
    Get a diverse sample of the dataframe by sampling from each title
    """
    sample_df = df.groupby(['title', 'is_impossible']).apply(lambda x: x.sample(min(len(x), max(1, sample_size // 50)), random_state=random_state)).reset_index(drop=True)
    
    if len(sample_df) < sample_size:
        remaining_sample_size = sample_size - len(sample_df)
        remaining_df = df.drop(sample_df.index).sample(remaining_sample_size, random_state=random_state)
        sample_df = pd.concat([sample_df, remaining_df]).sample(frac=1, random_state=random_state).reset_index(drop=True)

    return sample_df.sample(min(sample_size, len(sample_df)), random_state=random_state).reset_index(drop=True)

train_df = json_to_dataframe_with_titles(json.load(open(f"{os.environ['SCRATCH']}/data/training/train.json")))
val_df = json_to_dataframe_with_titles(json.load(open(f"{os.environ['SCRATCH']}/data/training/dev.json")))

df = get_diverse_sample(val_df, sample_size=20, random_state=42)
df

/tmp/ipykernel_1630306/3662158.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_df = df.groupby(['title', 'is_impossible']).apply(lambda x: x.sample(min(len(x), max(1, sample_size // 50)), random_state=random_state)).reset_index(drop=True)


,title,question,context,is_impossible,answers
0,"Fresno,_California",Who is Kearney Boulevard named after?,"The neighborhood includes Kearney Boulevard, n...",False,"[M. Theo Kearney, M. Theo Kearney, M. Theo Kea..."
1,1973_oil_crisis,By which year did Chrysler ended its full size...,"Federal safety standards, such as NHTSA Federa...",False,"[1981, 1981, 1981, 1981, 1981]"
2,Prime_number,What type of value would the Basel function ha...,The zeta function is closely related to prime ...,True,[]
3,Black_Death,What does Graham Twigg propose about the sprea...,A variety of alternatives to the Y. pestis hav...,False,"[a form of anthrax, was a form of anthrax, the..."
4,Scottish_Parliament,What consequence of establishing the Scottish ...,A procedural consequence of the establishment ...,False,[able to vote on domestic legislation that app...
5,Force,What is more fundamental than force in quanton...,"In modern particle physics, forces and the acc...",False,"[conservation of momentum, conservation of mom..."
6,Construction,What is a complex net of contracts and other l...,A construction project is a complex net of con...,False,"[A construction project, A construction projec..."
7,Imperialism,No imperialism was carried out using which met...,"""The word ‘empire’ comes from the Latin word i...",True,[]
8,Packet_switching,Are the sizes of packets variable?,Packet switching contrasts with another princi...,True,[]
9,Ctenophora,"Are ctenophores predators, vegetarian or paras...",Almost all ctenophores are predators – there a...,False,"[Almost all ctenophores are predators, predato..."


### Creating the Embeddings

We will generate embeddings for the entire training dataset. Our train dataset contains ~130k QA pairs. This step can also be replaced by using an out-of-the-box vector database. For example, you can follow one of [these cookbooks](https://github.com/openai/openai-cookbook/tree/main/examples/vector_databases) to set up your vector database. 

In [4]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('BAAI/bge-m3', cache_folder=f"{os.environ['SCRATCH']}/data/training/") 

def generate_embedding_from_dataframe(df: pd.DataFrame):
    batch_size = 64
    questions = df["question"].tolist()
    total_batches = len(questions) // batch_size + 1
    pbar = tqdm(total=len(questions), desc="Generating embeddings")
    embeddings = []
    for i in range(total_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(questions))
        batch = questions[start_idx:end_idx]
        batch_embeddings = embedding_model.encode(batch, batch_size=batch_size)
        embeddings.extend(batch_embeddings)
        pbar.update(len(batch))
        
    pbar.close()
    
    # Convert embeddings to list of lists
    embeddings_list = [embedding.tolist() for embedding in embeddings]
    
    # Create a temporary DataFrame to hold the embeddings and existing DataFrame columns
    temp_df = df.copy()
    temp_df["embeddings"] = embeddings_list
    temp_df["id"] = temp_df.index
    
    return temp_df

In [5]:
#embed_df = generate_embedding_from_dataframe(train_df)
#embed_df.to_csv(f"{os.environ['SCRATCH']}/data/training/train_data_with_embeddings.csv", index=False)
#print("Embeddings successfully stored in train_data_with_embeddings.csv")
embed_df = pd.read_csv(f"{os.environ['SCRATCH']}/data/training/train_data_with_embeddings.csv")
import ast
embed_df['embeddings'] = embed_df['embeddings'].apply(ast.literal_eval)

### Building the Matching Algorithm

In this section, we'll develop a cosine similarity retrieval algorithm to find similar questions in our dataframe. We'll utilize our custom cosine similarity function for this purpose. 

Here we aim to demonstrate that the matching algorithm can be tailored to meet specific requirements, such as a particular threshold or a specified number of matches returned.

The`find_similar_questions` function accepts four parameters:

- `embedding`: The embedding for which we want to find a match.
- `embeddings`: A list of embeddings to search through for the best matches.
- `threshold` : This parameter specifies the minimum similarity score for a match to be considered valid. A higher threshold results in closer (better) matches, while a lower threshold allows for more samples to be returned, though they may not be as closely matched to the initial embedding.
- `top_k`: This parameter determines the number of samples to return that exceed the given threshold. These will be the top-scoring matches for the provided embedding.

In [6]:
def cosine_similarity_manual(vec1, vec2):
    """Calculate the cosine similarity between two vectors."""
    vec1 = np.array(vec1, dtype=float)
    vec2 = np.array(vec2, dtype=float)


    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)


def find_similar_questions(input_embedding, embeddings, threshold=0, top_k=2):
    """Find the most similar questions based on cosine similarity."""
    
    # Calculate cosine similarity between the input embedding and all other embeddings
    similarities = [(index, cosine_similarity_manual(input_embedding, vec)) for index, vec in enumerate(embeddings)]
    
    # Filter out any similarities below the threshold
    filtered_similarities = [(index, sim) for index, sim in similarities if sim >= threshold]
    
    # Sort the filtered similarities by similarity score
    sorted_indices = sorted(filtered_similarities, key=lambda x: x[1], reverse=True)[:top_k]

    # Return the top-k most similar questions
    return sorted_indices


In [7]:
def find_matching_questions_with_rag(query, data_with_embeddings):

   """Take the input query and find the most similar question based on cosine similarity."""
   
   # Select the embeddings from the DataFrame.
   embeddings = data_with_embeddings['embeddings'].tolist()
   
   similar_questions = []
   # Generate the embedding for the input query
   input_embedding = batch_embeddings = embedding_model.encode(query)
   # Find the most similar questions based on cosine similarity
   similar_indices = find_similar_questions(input_embedding, embeddings, threshold=0)
   similar_questions += [data_with_embeddings.iloc[i[0]] for i in similar_indices]
    
   return similar_questions

Our main function get_prompt serves as the workhorse for generating prompts for RAG. It does this by retrieving similar questions with our search function. 

In [8]:
def get_orig_prompt(row):

    query, row_context = row["question"], row["context"]

    instruction = """Answer the following Question based on the Context only. Only answer from the Context. If you don't know the answer, say 'I don't know'.\n\n"""
    
    prompt = [{"role": "system", "content": instruction}] + [
        {
            "role": "user",
            "content": f"""Question: {query}\n\nContext: {row_context}\n\nAnswer:"""
        },
    ]
    return prompt


In [23]:
def get_rag_prompt(row):

    query, row_context = row["question"], row["context"]

    # Query for similar questions 
    q1 = find_matching_questions_with_rag(query, embed_df)

    instruction = """Answer the following Question based on the Context only. Only answer from the Context. If you don't know the answer, say 'I don't know'.\n\n"""
    # If there is a smilar question, add it to the prompt
    
    def q_to_prompt(q):
        question, context = q["question"], q["context"]
        answers_list = ast.literal_eval(q["answers"])        
        answer = answers_list[0] if answers_list else "I don't know"
        return [
            {
                "role": "user", 
                "content": f"""Question: {question}\n\nContext: {context}\n\nAnswer:"""
            },
            {"role": "assistant", "content": answer},
        ]

    rag_prompt = []
    for q in q1:
        rag_prompt += q_to_prompt(q)
    

    rag_prompt += [
        {
            "role": "user",
            "content": f"""Question: {query}\n\nContext: {row_context}\n\nAnswer:"""
        },
    ]

    rag_prompt = [{"role": "system", "content": instruction}] + rag_prompt
    return rag_prompt



In [24]:
val_sample = get_diverse_sample(val_df, sample_size=10, random_state=42)
val_sample["rag_prompt"] = val_sample.progress_apply(get_rag_prompt, axis=1)
val_sample["orig_prompt"] = val_sample.progress_apply(get_orig_prompt, axis=1)

/tmp/ipykernel_1630306/3662158.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_df = df.groupby(['title', 'is_impossible']).apply(lambda x: x.sample(min(len(x), max(1, sample_size // 50)), random_state=random_state)).reset_index(drop=True)
100%|██████████| 10/10 [00:00<00:00, 23198.58it/s]


In [25]:
print(val_sample["orig_prompt"].iloc[0])

[{'role': 'system', 'content': "Answer the following Question based on the Context only. Only answer from the Context. If you don't know the answer, say 'I don't know'.\n\n"}, {'role': 'user', 'content': 'Question: Who is Kearney Boulevard named after?\n\nContext: The neighborhood includes Kearney Boulevard, named after early 20th century entrepreneur and millionaire M. Theo Kearney, which extends from Fresno Street in Southwest Fresno about 20 mi (32 km) west to Kerman, California. A small, two-lane rural road for most of its length, Kearney Boulevard is lined with tall palm trees. The roughly half-mile stretch of Kearney Boulevard between Fresno Street and Thorne Ave was at one time the preferred neighborhood for Fresno\'s elite African-American families. Another section, Brookhaven, on the southern edge of the West Side south of Jensen and west of Elm, was given the name by the Fresno City Council in an effort to revitalize the neighborhood\'s image. The isolated subdivision was for

In [26]:
print(val_sample["rag_prompt"].iloc[0])

[{'role': 'system', 'content': "Answer the following Question based on the Context only. Only answer from the Context. If you don't know the answer, say 'I don't know'.\n\n"}, {'role': 'user', 'content': 'Question: Who is the Bronx named for?\n\nContext: The Bronx is named after Jonas Bronck who created the first settlement as part of the New Netherland colony in 1639. The native Lenape were displaced after 1643 by settlers. In the 19th and 20th centuries, the Bronx received many immigrant groups as it was transformed into an urban community, first from various European countries (particularly Ireland, Germany and Italy) and later from the Caribbean region (particularly Puerto Rico, Jamaica and the Dominican Republic), as well as African American migrants from the American South. This cultural mix has made the Bronx a wellspring of both Latin music and hip hop.\n\nAnswer:'}, {'role': 'assistant', 'content': 'Jonas Bronck'}, {'role': 'user', 'content': "Question: For whom was Houston na

### Inference with RAG prompt

We can use gpt-oss for inference. 

In [27]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

# Load the model 
model_kwargs = dict(attn_implementation="eager", torch_dtype="auto", use_cache=True, device_map="auto")
model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs).cuda()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [28]:
def generate_answer(messages):
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    gen_kwargs = {
        "max_new_tokens": 512, 
        "do_sample": True, 
        "temperature": 0.6, 
        "top_p": None, 
        "top_k": None
    }

    output_ids = model.generate(input_ids, **gen_kwargs)
    
    # Slice the output to exclude the original prompt tokens
    input_length = input_ids.shape[1]
    generated_ids = output_ids[0][input_length:]
    
    raw_response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    if "<|assistantfinal|>" in raw_response:
        final_answer = raw_response.split("<|assistantfinal|>")[-1]
    elif "assistantfinal" in raw_response: 
        final_answer = raw_response.split("assistantfinal")[-1]
    else:
        final_answer = raw_response 
        
    final_answer = final_answer.replace(tokenizer.eos_token, "").strip()
    return final_answer
    
val_sample["rag_generated_answer"] = val_sample["rag_prompt"].progress_apply(generate_answer)
val_sample["generated_answer"] = val_sample["orig_prompt"].progress_apply(generate_answer)
#val_sample["generated_answer"] = val_sample["few_shot_prompt"].progress_apply(generate_answer)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
100%|██████████| 10/10 [01:43<00:00, 10.40s/it]


In [29]:
# Check the results
pd.set_option('display.max_colwidth', None)
print(val_sample[["question", "generated_answer", "rag_generated_answer"]].iloc[0])

question                                                                                Who is Kearney Boulevard named after?
generated_answer        Kearney Boulevard is named after early‑20th‑century entrepreneur and millionaire **M. Theo Kearney**.
rag_generated_answer                                                                                         M. Theo Kearney.
Name: 0, dtype: object


In [16]:
# Optional: Save and load to cache the results so you don't have to rerun generation
# val_sample.to_json(f"{os.environ['SCRATCH']}/data/training/val_sample_results.json", orient="records", lines=True)
# val_sample = pd.read_json(f"{os.environ['SCRATCH']}/data/training/val_sample_results.json}", orient="records", lines=True)


### Evaluation

Tomorrow